In [33]:
# Load geojson as dataframe
import geopandas as gpd

gdf = gpd.read_file("final/final_output.geojson")

In [45]:
import json
from typing import Dict

import pandas as pd

col_descs = {
  "county_fips": {
    "description": "5-digit FIPS code for the county",
    "datatype": "integer",
  },
  "state": {
    "description": "State name",
    "datatype": "string",
  },
  "county": {
    "description": "County name",
    "datatype": "string",
  },
  "year": {
    "description": "Year",
    "datatype": "integer",
  },
  "total_population": {
    "description": "Total population",
    "datatype": "integer",
  },
  "population_below_poverty": {
    "description": "Population below poverty",
    "datatype": "integer",
  },
  "population_below_poverty_percent": {
    "description": "Population below poverty percent",
    "datatype": "float",
  },
  "adults_25_and_older_less_than_high_school_graduate": {
    "description": "Adults 25 and older less than high school graduate",
    "datatype": "integer",
  },
  "adults_25_and_older_less_than_high_school_graduate_percent": {
    "description": "Adults 25 and older less than high school graduate percent",
    "datatype": "float",
  },
  "adults_25_and_older_with_bachelor's_degree_or_higher": {
    "description": "Adults 25 and older with bachelor's degree or higher",
    "datatype": "integer",
  },
  "adults_25_and_older_with_bachelor's_degree_or_higher_percent": {
    "description": "Adults 25 and older with bachelor's degree or higher percent",
    "datatype": "float",
  },
  "total_investment_dollars": {
    "description": "Total investment dollars",
    "datatype": "integer",
  },
  "total_number_of_investments": {
    "description": "Total number of investments",
    "datatype": "integer",
  },
  "number_of_households": {
    "description": "Number of households",
    "datatype": "integer",
  },
  "average_income_per_household": {
    "description": "Average income per household",
    "datatype": "float",
  },
  "total_median_earnings": {
    "description": "Total median earnings",
    "datatype": "float",
  },
  "geometry": {
    "description": "Geometry",
    "datatype": "geometry",
  },
}

# Ensure each column in gdf is the correct datatype as specified, and drop unnecessary columns

# First, build a mapping from column name to its intended datatype from the col_descs dictionary above:
col_datatypes = {k: v["datatype"] for k, v in col_descs.items()}

# Keep only columns listed in col_descs
gdf = gdf[[col for col in gdf.columns if col in col_descs]]

# Ensure correct types for each column
for col, dtype in col_datatypes.items():
    if col not in gdf.columns:
        continue
    # Map datatype field to pandas dtype
    if dtype == "integer":
        # Use pandas' nullable integer (Int64) to allow NaNs if column is nullable
        gdf[col] = gdf[col].astype("Int64")
    elif dtype == "float":
        gdf[col] = gdf[col].astype("float64")
    elif dtype == "string":
        gdf[col] = gdf[col].astype("string")
    elif dtype == "boolean":
        gdf[col] = gdf[col].astype("boolean")
    # geometry is handled automatically by geopandas


print(gdf.dtypes)


def normalize_example(x):
    # Converts numpy scalars to float/int, leaves normal Python types unchanged;
    # for non-numerics, leave unchanged except for numpy objects (e.g. np.str_)
    # Special handling for geometry objects: leave as-is
    # Also, convert pd.Timestamp/numpy datetime to ISO string
    import numpy as np
    import pandas as pd

    if isinstance(x, (np.generic,)):
        if np.issubdtype(type(x), np.floating):
            return float(x)
        elif np.issubdtype(type(x), np.integer):
            return int(x)
        elif np.issubdtype(type(x), np.bool_):
            return bool(x)
        elif np.issubdtype(type(x), np.str_):
            return str(x)
        elif np.issubdtype(type(x), np.datetime64):
            return pd.Timestamp(x).isoformat()
        else:
            return x
    elif isinstance(x, pd.Timestamp):
        return x.isoformat()
    else:
        return x


def create_data_dictionary(gdf: gpd.GeoDataFrame, col_descs: Dict[str, str]):
    data_dictionary = []
    # Only use columns that are present in col_descs
    for col in col_descs.keys():
        if col not in gdf.columns:
            continue  # Skip if column is not in the GeoDataFrame
        
        series = gdf[col]
        dtype = str(series.dtype)
        # Nullable
        nullable = series.isnull().any()
        num_unique_values = series.nunique(dropna=True)
        num_null_values = int(series.isnull().sum())
        desc = col_descs[col]["description"]
        human_name = col.replace("_", " ").title()

        print(col, dtype)

        # Determine datatype similar to ColumnDatatype
        col_dtype = series.dtype

        if pd.api.types.is_integer_dtype(col_dtype):
            datatype = "integer"
        elif pd.api.types.is_float_dtype(col_dtype):
            datatype = "float"
        elif pd.api.types.is_bool_dtype(col_dtype):
            datatype = "boolean"
        elif col_dtype.name == "geometry":
            # handle geometry differently
            col_info = {
                "id": col,
                "name": col,
                "description": desc,
                "datatype": "geometry",
                "nullable": bool(nullable),
                "humanReadableName": human_name,
                "numUniqueValues": int(num_unique_values) if num_unique_values is not None else None,
                "numNullValues": int(num_null_values) if num_null_values is not None else None,
            }

            data_dictionary.append(col_info)
            continue
        else:
            datatype = "string"

        # min/max/length/unit
        min_val = None
        max_val = None
        length_val = None
        unit_val = None
        example_values = None
        possible_values = None

        if datatype in ("integer", "float"):
            if not series.empty:
                min_val = float(series.min()) if series.notnull().any() else None
                max_val = float(series.max()) if series.notnull().any() else None
            # Optionally: units if can be inferred from description
            desc_lc = col_descs[col]["description"].lower()
            
            if "percent" in desc_lc:
                unit_val = "percent"
            elif "usd" in desc_lc or "dollars" in desc_lc or "income" in desc_lc or "earnings" in desc_lc:
                unit_val = "USD"

        elif datatype == "string":
            try:
                length_val = int(series.map(lambda x: len(x) if isinstance(x, str) else 0).max())
            except Exception:
                length_val = None
            # possibleValues: for a reasonable string col
            n_unique = series.nunique(dropna=True)
            if n_unique > 0 and n_unique <= 20:
                possible_values = sorted([normalize_example(v) for v in series.dropna().unique()])
        elif datatype == "boolean":
            possible_values = [True, False]
        # Geometry typically no extra stats

        # numUnique/Null
        num_unique_values = series.nunique(dropna=True)
        num_null_values = int(series.isnull().sum())
        # Sample example values (show up to 5, if exist), normalize types
        if not series.dropna().empty:
            # list(series.dropna().unique()[:5]) —> need to convert np types to normal float/int
            examples = list(series.dropna().unique()[:5])
            example_values = [normalize_example(v) for v in examples]
        else:
            example_values = None


        # Compose full dictionary
        col_info = {
            "id": col,
            "name": col,
            "description": col_descs[col]["description"],
            "datatype": datatype,
            "nullable": bool(nullable),
            "humanReadableName": human_name,
            "min": min_val,
            "max": max_val,
            "unit": unit_val,
            "length": length_val,
            "possibleValues": possible_values,
            "exampleValues": example_values,
            "numUniqueValues": int(num_unique_values) if num_unique_values is not None else None,
            "numNullValues": int(num_null_values) if num_null_values is not None else None,
        }

        data_dictionary.append(col_info)

    return data_dictionary


col_descs = create_data_dictionary(gdf, col_descs)

data_dict = {
  "id": "fahe-502-investments",
  "kind": "geospatial",
  "location": "internal",
  "name": "FAHE 502 Investments",
  "description": "FAHE 502 investment data by county with demographic and economic indicators",
  "columns": col_descs
}

with open("workspace/502-investments/dictionary.json", "w") as f:
    json.dump(data_dict, f, indent=2)

# TODO: create filtered geoparquets
# TODO: create data dictionaries

county_fips                                                              Int64
state                                                           string[python]
county                                                          string[python]
year                                                                     Int64
total_population                                                         Int64
population_below_poverty                                                 Int64
population_below_poverty_percent                                       float64
adults_25_and_older_less_than_high_school_graduate                       Int64
adults_25_and_older_less_than_high_school_graduate_percent             float64
adults_25_and_older_with_bachelor's_degree_or_higher                     Int64
adults_25_and_older_with_bachelor's_degree_or_higher_percent           float64
total_investment_dollars                                                 Int64
total_number_of_investments                         

In [14]:
import pandas as pd

gdf.head()

# Figure out quintiles to color by
# get the investments per capita (total_investment_dollars / total_population)
gdf["investment_dollars_per_capita"] = gdf["total_investment_dollars"] / gdf["total_population"]

# get the quintile breakpoints
quintile_breaks = gdf["investment_dollars_per_capita"].quantile([0.2, 0.4, 0.6, 0.8]).values

# assign quintile labels (1-5) to each row
gdf["investment_dollars_per_capita_quintile"] = pd.cut(
    gdf["investment_dollars_per_capita"],
    bins=[-float("inf")] + list(quintile_breaks) + [float("inf")],
    labels=[1, 2, 3, 4, 5]
)

# print the quintile breakpoints and head
print("Quintile breakpoints:", quintile_breaks)
print(gdf[["investment_dollars_per_capita", "investment_dollars_per_capita_quintile"]].head())


Quintile breakpoints: [ 45.30702375  81.05456271 121.6377597  191.98828354]
   investment_dollars_per_capita investment_dollars_per_capita_quintile
0                      61.341118                                      2
1                      31.978595                                      1
2                      83.196838                                      3
3                     108.439857                                      3
4                      77.374758                                      2


In [16]:
# Do kmeans clustering to get 5 clusters
from sklearn.cluster import KMeans

# Drop rows with NaN in 'investment_dollars_per_capita' before clustering
gdf_nonan = gdf.dropna(subset=["investment_dollars_per_capita"]).copy()

kmeans = KMeans(n_clusters=5, random_state=42)
gdf_nonan["cluster"] = kmeans.fit_predict(gdf_nonan[["investment_dollars_per_capita"]])

# Merge the cluster labels back into the original gdf (NaNs will remain for rows that were dropped)
gdf["cluster"] = gdf_nonan["cluster"]

# print the cluster labels and head
print("Cluster labels:", gdf["cluster"].dropna().unique())
print(gdf[["investment_dollars_per_capita", "cluster"]].head())

Cluster labels: [0. 2. 1. 4. 3.]
   investment_dollars_per_capita  cluster
0                      61.341118      0.0
1                      31.978595      0.0
2                      83.196838      0.0
3                     108.439857      2.0
4                      77.374758      0.0


In [41]:
# Run tippacanoe to generate pmtiles from geojson
# May need to cast ints to floats
import subprocess

# Convert gdf to FlatGeobuf format (.fgb)
gdf.to_file("workspace/502-investments/layers/layer.fgb", driver="FlatGeobuf")


cmd = [
  "tippecanoe",
  "-o",
  "workspace/502-investments/layers/layer.pmtiles",
  "-zg",
  "--drop-densest-as-needed",
  "--extend-zooms-if-still-dropping",
  "-l",
  "fgb",
  "workspace/502-investments/layers/layer.fgb"
]

subprocess.run(cmd, check=True)

detected indexed FlatGeobuf: assigning feature IDs by sequence
8717 features, 4827504 bytes of geometry and attributes, 229261 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
Choosing a maxzoom of -z1 for features typically 148605 feet (45295 meters) apart, and at least 62723 feet (19118 meters) apart
Choosing a maxzoom of -z5 for resolution of about 9094 feet (2772 meters) within features
  99.9%  5/8/12  


CompletedProcess(args=['tippecanoe', '-o', 'workspace/502-investments/layers/layer.pmtiles', '-zg', '--drop-densest-as-needed', '--extend-zooms-if-still-dropping', '-l', 'fgb', 'workspace/502-investments/layers/layer.fgb'], returncode=0)

In [40]:
# Create the geoparquet
gdf.to_parquet("workspace/502-investments/data/block-0.parquet")

In [17]:
# Copy the geojson to the public folder
import shutil

shutil.copy("final/final_output.geojson", "502-investments-map/public/502-investments.geojson")


'502-investments-map/public/502-investments.geojson'